In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG crypto_pipeline")

DataFrame[]

In [0]:
# Databricks notebook source
# ==========================================================
# Bronze Layer
# Notebook: 01_ingest_coin_market_data
# Purpose: Pull latest cryptocurrency market data from CoinGecko API and append it to the Bronze Delta table.
# Layer: Bronze
# Write Mode: Append Only
# ==========================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import Row
from datetime import datetime
import requests
import uuid
import time
# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
CATALOG = "crypto_pipeline"
BRONZE_TABLE = f"{CATALOG}.bronze.raw_coin_market_data"
LOG_TABLE = f"{CATALOG}.meta.ingestion_log"
API_URL = (
    "https://api.coingecko.com/api/v3/coins/markets"
    "?vs_currency=usd"
    "&order=market_cap_desc"
    "&per_page=50"
    "&page=1"
)
MAX_RETRIES = 3
RETRY_DELAY = 5
# ----------------------------------------------------------
# Use Catalog
# ----------------------------------------------------------
spark.sql(f"USE CATALOG {CATALOG}")
# ----------------------------------------------------------
# Create Metadata Table if it does not exist
# ----------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE}
(
    batch_id STRING,
    source_run_id STRING,
    ingestion_timestamp TIMESTAMP,
    status STRING,
    rows_ingested INT,
    error_message STRING
)
USING DELTA
""")
# ----------------------------------------------------------
# Create Bronze Table if not exists
# ----------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE}
(
id STRING,
symbol STRING,
name STRING,
image STRING,
current_price DOUBLE,
market_cap DOUBLE,
market_cap_rank INT,
fully_diluted_valuation DOUBLE,
total_volume DOUBLE,
high_24h DOUBLE,
low_24h DOUBLE,
price_change_24h DOUBLE,
price_change_percentage_24h DOUBLE,
market_cap_change_24h DOUBLE,
market_cap_change_percentage_24h DOUBLE,
circulating_supply DOUBLE,
total_supply DOUBLE,
max_supply DOUBLE,
ath DOUBLE,
ath_change_percentage DOUBLE,
ath_date STRING,
atl DOUBLE,
atl_change_percentage DOUBLE,
atl_date STRING,
roi STRING,
last_updated STRING,
_ingested_at TIMESTAMP,
_batch_id STRING,
_source_run_id STRING
)
USING DELTA
""")
# ----------------------------------------------------------
# Metadata
# ----------------------------------------------------------
batch_id = str(uuid.uuid4())
source_run_id = str(uuid.uuid4())
from datetime import datetime, UTC
ingested_at = datetime.now(UTC)
# ----------------------------------------------------------
# Function to call API
# ----------------------------------------------------------
def fetch_data():
    response = requests.get(API_URL, timeout=60)
    response.raise_for_status()
    return response.json()
# ----------------------------------------------------------
# Retry Logic
# ----------------------------------------------------------
api_data = None
error_message = None
for attempt in range(MAX_RETRIES):
    try:
        api_data = fetch_data()
        break
    except Exception as e:
        error_message = str(e)
        print(f"Attempt {attempt+1} failed")
        if attempt < MAX_RETRIES - 1:
            time.sleep(RETRY_DELAY)
# ----------------------------------------------------------
# If API Failed
# ----------------------------------------------------------
if api_data is None:
    log = [
        Row(
            batch_id=batch_id,
            source_run_id=source_run_id,
            ingestion_timestamp=ingested_at,
            status="FAILED",
            rows_ingested=0,
            error_message=error_message
        )
    ]
    spark.createDataFrame(log).write.mode("append").saveAsTable(LOG_TABLE)
    raise Exception(error_message)
# ----------------------------------------------------------
# Convert API Response
# ----------------------------------------------------------
rows = []
for coin in api_data:
    rows.append(
        Row(
            id=coin.get("id"),
            symbol=coin.get("symbol"),
            name=coin.get("name"),
            image=coin.get("image"),
            current_price=coin.get("current_price"),
            market_cap=coin.get("market_cap"),
            market_cap_rank=coin.get("market_cap_rank"),
            fully_diluted_valuation=coin.get("fully_diluted_valuation"),
            total_volume=coin.get("total_volume"),
            high_24h=coin.get("high_24h"),
            low_24h=coin.get("low_24h"),
            price_change_24h=coin.get("price_change_24h"),
            price_change_percentage_24h=coin.get("price_change_percentage_24h"),
            market_cap_change_24h=coin.get("market_cap_change_24h"),
            market_cap_change_percentage_24h=coin.get("market_cap_change_percentage_24h"),
            circulating_supply=coin.get("circulating_supply"),
            total_supply=coin.get("total_supply"),
            max_supply=coin.get("max_supply"),
            ath=coin.get("ath"),
            ath_change_percentage=coin.get("ath_change_percentage"),
            ath_date=coin.get("ath_date"),
            atl=coin.get("atl"),
            atl_change_percentage=coin.get("atl_change_percentage"),
            atl_date=coin.get("atl_date"),
            roi=str(coin.get("roi")),
            last_updated=coin.get("last_updated"),
            _ingested_at=ingested_at,
            _batch_id=batch_id,
            _source_run_id=source_run_id
        )
    )
# ----------------------------------------------------------
# Create Spark DataFrame
# ----------------------------------------------------------
schema = StructType([
    StructField("id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("name", StringType(), True),
    StructField("image", StringType(), True),
    StructField("current_price", DoubleType(), True),
    StructField("market_cap", DoubleType(), True),
    StructField("market_cap_rank", IntegerType(), True),
    StructField("fully_diluted_valuation", DoubleType(), True),
    StructField("total_volume", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("price_change_24h", DoubleType(), True),
    StructField("price_change_percentage_24h", DoubleType(), True),
    StructField("market_cap_change_24h", DoubleType(), True),
    StructField("market_cap_change_percentage_24h", DoubleType(), True),
    StructField("circulating_supply", DoubleType(), True),
    StructField("total_supply", DoubleType(), True),
    StructField("max_supply", DoubleType(), True),
    StructField("ath", DoubleType(), True),
    StructField("ath_change_percentage", DoubleType(), True),
    StructField("ath_date", StringType(), True),
    StructField("atl", DoubleType(), True),
    StructField("atl_change_percentage", DoubleType(), True),
    StructField("atl_date", StringType(), True),
    StructField("roi", StringType(), True),
    StructField("last_updated", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_batch_id", StringType(), True),
    StructField("_source_run_id", StringType(), True)
])
bronze_df = spark.createDataFrame(rows, schema=schema)
# ----------------------------------------------------------
# Append Only
# ----------------------------------------------------------
bronze_df.write.mode("append").saveAsTable(BRONZE_TABLE)
# ----------------------------------------------------------
# Log Success
# ----------------------------------------------------------
log = [
    Row(
        batch_id=batch_id,
        source_run_id=source_run_id,
        ingestion_timestamp=ingested_at,
        status="SUCCESS",
        rows_ingested=bronze_df.count(),
        error_message=None
    )
]
spark.createDataFrame(log, schema=StructType([StructField("batch_id", StringType(), True), StructField("source_run_id", StringType(), True), StructField("ingestion_timestamp", TimestampType(), True), StructField("status", StringType(), True), StructField("rows_ingested", IntegerType(), True), StructField("error_message", StringType(), True)])).write.mode("append").saveAsTable(LOG_TABLE)
# ----------------------------------------------------------
# Display
# ----------------------------------------------------------
print("Bronze Ingestion Complete")
print(f"Rows: {bronze_df.count()}")
print(f"Batch ID: {batch_id}")
display(bronze_df)

Bronze Ingestion Complete
Rows: 50
Batch ID: c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475


id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,low_24h,price_change_24h,price_change_percentage_24h,market_cap_change_24h,market_cap_change_percentage_24h,circulating_supply,total_supply,max_supply,ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated,_ingested_at,_batch_id,_source_run_id
bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,65922.0,1.322409077378E12,1,1.322409077378E12,3.2062806671E10,66840.0,65709.0,-251.1924731802137,-0.3796,-4.869795748832764E9,-0.3669,2.0059706E7,2.0059706E7,2.1E7,126080.0,-47.7142,2025-10-06T18:57:42.558Z,67.81,97116.99771,2013-07-06T00:00:00.000Z,None,2026-07-22T08:57:59.987Z,2026-07-22T08:59:21.134Z,c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475,8403e77b-2361-4dba-b4f5-99baf17ee51c
ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images/279/large/ethereum.png?1696501628,1919.35,2.31629638606E11,2,2.31629638606E11,1.0336311395E10,1944.33,1910.77,-18.99310764918164,-0.97986,-2.2833990939561462E9,-0.97617,1.206827757241115E8,1.206827757241115E8,null,4946.05,-61.19431,2025-08-24T19:21:03.333Z,0.432979,443189.13732,2015-10-20T00:00:00.000Z,"{'times': 37.92309041874939, 'currency': 'btc', 'percentage': 3792.3090418749393}",2026-07-22T08:57:58.434Z,2026-07-22T08:59:21.134Z,c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475,8403e77b-2361-4dba-b4f5-99baf17ee51c
tether,usdt,Tether,https://coin-images.coingecko.com/coins/images/325/large/Tether.png?1696501661,0.999272,1.84091192562E11,3,1.89554140013E11,4.8507675246E10,0.999442,0.999134,8.548E-5,0.00855,2.1604708E7,0.01174,1.842257944663005E11,1.896927362591762E11,null,1.32,-24.47567,2018-07-24T00:00:00.000Z,0.572521,74.53662,2015-03-02T00:00:00.000Z,None,2026-07-22T08:57:48.727Z,2026-07-22T08:59:21.134Z,c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475,8403e77b-2361-4dba-b4f5-99baf17ee51c
binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images/825/large/bnb-icon2_2x.png?1696501970,569.71,7.587212412E10,4,7.587212412E10,5.64901171E8,578.99,567.24,-7.535549706710867,-1.30542,-1.0028850423137207E9,-1.30457,1.3316585956E8,1.3316585956E8,2.0E8,1369.99,-58.41479,2025-10-13T08:41:24.131Z,0.0398177,1430705.66251,2017-10-19T00:00:00.000Z,None,2026-07-22T08:58:01.243Z,2026-07-22T08:59:21.134Z,c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475,8403e77b-2361-4dba-b4f5-99baf17ee51c
usd-coin,usdc,USDC,https://coin-images.coingecko.com/coins/images/6319/large/USDC.png?1769615602,1.0,7.3198530272E10,5,7.3198504606E10,1.1191746137E10,1.0,0.999727,3.4142E-4,0.03415,4.6604775E7,0.06371,7.318497485919116E10,7.318494919762967E10,null,1.043,-4.17665,2018-11-15T00:00:00.000Z,0.877647,13.92771,2023-03-11T08:02:13.981Z,None,2026-07-22T08:57:55.180Z,2026-07-22T08:59:21.134Z,c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475,8403e77b-2361-4dba-b4f5-99baf17ee51c
ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images/44/large/xrp-symbol-white-128.png?1696501442,1.13,7.0829003636E10,6,1.13370891882E11,1.35471518E9,1.16,1.13,9.2615E-4,0.08175,8.6454855E7,0.12221,6.2466503703E10,9.9985639695E10,1.0E11,3.65,-68.90372,2025-07-18T03:40:53.808Z,0.00268621,42111.49792,2014-05-22T00:00:00.000Z,None,2026-07-22T08:57:51.744Z,2026-07-22T08:59:21.134Z,c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475,8403e77b-2361-4dba-b4f5-99baf17ee51c
solana,sol,Solana,https://coin-images.coingecko.com/coins/images/4128/large/solana.png?1718769756,77.34,4.507139695E10,7,4.8783134464E10,1.496050329E9,78.51,77.01,-0.9703418514234698,-1.23907,-5.723200401762085E8,-1.25389,5.82749377312526E8,6.307401845925688E8,null,293.31,-73.63149,2025-01-19T11:15:27.957Z,0.500801,15343.65022,2020-05-11T19:35:23.449Z,None,2026-07-22T08:57:51.485Z,2026-07-22T08:59:21.134Z,c9d8b898-a1b6-41c7-9f5b-3b6bd9dfb475,8403e77b-2361-4dba-b4f5-99baf17ee51c
tron,trx,TRON,https://coin-images.coingecko.com/coins/images/1094/large/photo_2026-04-13_09-59-16.png?1776048311,0.328866,3.1200183732E10,8,3.1200589931E10,3.25557615E8,0.329668,0.326312,0.0